# Fusion — Real Inference-Time Score Combiner

In [1]:
import sys
sys.path.insert(0, r"C:\FYP\src")
from utils.config import (
    TABULAR_CLEAN_PATH, CACHE_IMAGES_PATH, CACHE_MANIFEST_PATH, IMAGING_OOF_PREDICTIONS_PATH,
    CHECKPOINTS_CLINICAL_FINAL_DIR, CHECKPOINTS_IMAGING_FINAL_DIR, CHECKPOINTS_FUSION_FINAL_DIR,
    FUSION_IMAGING_AGGREGATION_PATH, RANDOM_SEED, ensure_dirs,
)
from imaging.models import ResNet50UNet

import json
import textwrap
from datetime import date

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    brier_score_loss, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)

ensure_dirs()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Imports OK, device: {DEVICE}")

Imports OK, device: cuda


## The Two Fixed Rules — Named Constants

In [2]:
W_IMAGING = 0.4
W_TABULAR = 0.6
assert abs((W_IMAGING + W_TABULAR) - 1.0) < 1e-9, "Weights must sum to 1.0"

COMBINATION_RULE_REASONING = (
    "A deliberate, documented judgment call, NOT an empirically fitted parameter -- there is no "
    "paired ground truth (no patient has both a CT scan and a urine sample) to fit or tune weights "
    "against. Imaging is weighted BELOW clinical (0.4 vs 0.6) because the imaging branch carries a "
    "disclosed, unresolved confound-check finding -- its detection confidence is not yet confirmed "
    "to reflect tumour-specific reasoning -- so it is deliberately not allowed to dominate a "
    "disagreement between the two branches. The discount is kept modest (0.4, not lower) because "
    "imaging's own calibration is strong and, post-calibration, it is slightly under-confident, so "
    "a larger discount would over-correct."
)

IMAGING_SLICE_AGGREGATION = "mean"
IMAGING_VOLUME_BATCH_SIZE = 32  # matches the training batch size; keeps a ~250-slice volume off the GPU all at once

IMAGING_SLICE_AGGREGATION_REASONING = (
    "Mean of the per-slice calibrated probabilities across a whole CT volume. A fixed, hand-set "
    "inference-time rule, NOT a fitted parameter -- the imaging branch was trained and evaluated "
    "purely slice-level and is not retrofitted. Chosen because a real CT scan is a volume "
    "(128-526 slices in this dataset), so requiring a user to pre-select the single slice showing "
    "the tumour would push the model's own detection job onto them. 'mean' specifically beat "
    "median/max/top-k on ROC-AUC, F1 and Brier when measured against imaging's real 5-fold OOF "
    "predictions (see results/fusion/imaging_slice_vs_volume_aggregation.csv): max and top-k buy "
    "marginally higher recall but collapse precision (~27% false-positive rate on healthy patients), "
    "which is not a worthwhile trade given the disclosed confound. IMPORTANT INTERPRETIVE CAVEAT: "
    "mean winning is itself partly a fingerprint of that confound -- within a single cancer patient "
    "the model's per-slice scores barely vary (std ~0.05, and even the lowest-scoring slice averages "
    "~0.42) despite only a minority of that patient's slices actually containing tumour. A genuinely "
    "tumour-localizing model would show high within-patient variance and would favour max/top-k. "
    "Volume aggregation therefore improves the numbers but does NOT resolve, and can visually "
    "obscure, the tumour-under-reliance finding."
)

print(f"W_IMAGING = {W_IMAGING}, W_TABULAR = {W_TABULAR} (sum = {W_IMAGING + W_TABULAR})")
print(f"IMAGING_SLICE_AGGREGATION = '{IMAGING_SLICE_AGGREGATION}' (batch size {IMAGING_VOLUME_BATCH_SIZE})")

W_IMAGING = 0.4, W_TABULAR = 0.6 (sum = 1.0)
IMAGING_SLICE_AGGREGATION = 'mean' (batch size 32)


## Load the Clinical Branch — `checkpoints/clinical/final/`

In [3]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge


class MICE_CA19_9Imputer:
    PREDICTORS = ["creatinine", "LYVE1", "REG1B", "TFF1", "age"]
    TARGET = "plasma_CA19_9"

    def __init__(self, random_state=RANDOM_SEED):
        self.imputer = IterativeImputer(estimator=BayesianRidge(), random_state=random_state)

    def fit(self, train_df):
        self.imputer.fit(train_df[self.PREDICTORS + [self.TARGET]])
        return self

    def transform(self, target_df):
        out = target_df.copy()
        out[self.TARGET] = self.imputer.transform(out[self.PREDICTORS + [self.TARGET]])[:, -1]
        return out


TABULAR_FEATURES = ["creatinine", "LYVE1", "REG1B", "TFF1", "plasma_CA19_9", "age", "sex"]

clinical_model = joblib.load(CHECKPOINTS_CLINICAL_FINAL_DIR / "model.pkl")
clinical_imputer = joblib.load(CHECKPOINTS_CLINICAL_FINAL_DIR / "ca19_9_imputer.pkl")
clinical_calibrator = joblib.load(CHECKPOINTS_CLINICAL_FINAL_DIR / "calibrator.pkl")

with open(CHECKPOINTS_CLINICAL_FINAL_DIR / "model_card.json") as f:
    clinical_model_card = json.load(f)

print(
    f"Clinical branch loaded: {clinical_model_card['model']}, "
    f"trained_on={clinical_model_card['trained_on']}, n_patients={clinical_model_card['n_patients']}"
)

Clinical branch loaded: xgboost, trained_on=2026-07-16, n_patients=590


## Load the Imaging Branch — `checkpoints/imaging/final/`

In [4]:
with open(CHECKPOINTS_IMAGING_FINAL_DIR / "pod_training_run.json") as f:
    imaging_pod_run = json.load(f)
IMAGING_BOX_SIZE = imaging_pod_run["box_size"]

imaging_state_dict = torch.load(CHECKPOINTS_IMAGING_FINAL_DIR / "model.pt", map_location=DEVICE, weights_only=True)
bad_keys = [k for k in imaging_state_dict if k.startswith("_orig_mod.")]
assert not bad_keys, f"torch.compile prefix leakage: {bad_keys[:5]}"

imaging_model = ResNet50UNet(pretrained=False).to(DEVICE)
imaging_model.load_state_dict(imaging_state_dict)
imaging_model.eval()

imaging_calibrator = joblib.load(CHECKPOINTS_IMAGING_FINAL_DIR / "calibrator.pkl")

with open(CHECKPOINTS_IMAGING_FINAL_DIR / "model_card.json") as f:
    imaging_model_card = json.load(f)

IMAGING_CAVEAT = (
    "KNOWN LIMITATION (checkpoints/imaging/final/model_card.json, confound-check investigation): "
    + imaging_model_card["known_limitations"]["confound_check_summary"]
)

print(f"Imaging branch loaded: {imaging_model_card['candidate']}, box_size={IMAGING_BOX_SIZE}, device={DEVICE}")
print("\nIMAGING CAVEAT:")
print(textwrap.fill(IMAGING_CAVEAT, 100))

Imaging branch loaded: resnet50_unet, box_size=320, device=cuda

IMAGING CAVEAT:
KNOWN LIMITATION (checkpoints/imaging/final/model_card.json, confound-check investigation): A
rigorous, 6-round confound-check investigation (Grad-CAM + 5 other attribution methods, then causal
occlusion testing) found this architecture's detection head originally relied MORE on the BOX=320
packing's synthetic padding than on the real tumour region (Round 4: 2.03x a random-patch control,
p=0.044). A random-resized-crop augmentation (augment=True, used in this final fit) verified-fixed
the padding shortcut (Round 5: 1.08x control, p=0.917 -- statistically indistinguishable from the
control). A second remediation attempt (Round 6: mask-preserving random erase) targeting the model's
remaining UNDER-reliance on the tumour region itself did NOT help (0.58x control, p=0.0148,
statistically unchanged from Round 5's 0.62x) and was reverted -- this promoted model does not
include that change. TUMOUR UNDER-RELIANCE 

## Measurement: Slice-Level vs. Volume-Level Aggregation

In [5]:
imaging_oof = pd.read_csv(IMAGING_OOF_PREDICTIONS_PATH)
imaging_oof = imaging_oof[imaging_oof["candidate"] == imaging_model_card["candidate"]].copy()
imaging_oof["p"] = imaging_calibrator.predict_proba(imaging_oof[["y_proba"]].to_numpy())[:, 1]

# Leakage guard: patient-level aggregation is only valid if no patient's slices span folds.
assert (imaging_oof.groupby("patient_id")["fold"].nunique() > 1).sum() == 0, \
    "Some patients span multiple folds -- patient-level OOF aggregation would mix train/held-out predictions"
assert (imaging_oof.groupby("patient_id")["y_true"].nunique() > 1).sum() == 0, \
    "Some patients have inconsistent slice labels -- patient-level label is ill-defined"

patient_labels = imaging_oof.groupby("patient_id")["y_true"].first()
slice_counts = imaging_oof.groupby("patient_id").size()
print(f"{len(imaging_oof):,} OOF slices over {imaging_oof['patient_id'].nunique()} patients "
      f"(slices/patient: min={slice_counts.min()}, median={int(slice_counts.median())}, max={slice_counts.max()})")
print("Fold-disjointness and label-consistency checks passed.\n")


def score_rows(granularity, rule, y_true, y_score, threshold=0.5):
    y_true, y_score = np.asarray(y_true), np.asarray(y_score)
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "granularity": granularity, "rule": rule, "n_units": len(y_true),
        "roc_auc": roc_auc_score(y_true, y_score),
        "precision": precision_score(y_true, y_pred, zero_division=np.nan),
        "recall": recall_score(y_true, y_pred, zero_division=np.nan),
        "f1": f1_score(y_true, y_pred, zero_division=np.nan),
        "brier": brier_score_loss(y_true, y_score),
        "confusion_matrix_tn_fp_fn_tp": f"{tn}/{fp}/{fn}/{tp}",
    }


grouped = imaging_oof.groupby("patient_id")["p"]
candidate_aggregations = {
    "mean": grouped.mean(),
    "median": grouped.median(),
    "max": grouped.max(),
    "top-5%-slice-mean": grouped.apply(lambda s: s.nlargest(max(1, int(round(len(s) * 0.05)))).mean()),
    "top-10-slice-mean": grouped.apply(lambda s: s.nlargest(min(10, len(s))).mean()),
    "90th-percentile": grouped.apply(lambda s: float(np.percentile(s, 90))),
}

rows = [score_rows("slice (baseline)", "none -- every slice scored independently",
                   imaging_oof["y_true"], imaging_oof["p"])]
rows += [score_rows("volume (patient)", rule, patient_labels.loc[agg.index].values, agg.values)
         for rule, agg in candidate_aggregations.items()]

aggregation_df = pd.DataFrame(rows)
aggregation_df.to_csv(FUSION_IMAGING_AGGREGATION_PATH, index=False)

print(aggregation_df.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"\nSaved {FUSION_IMAGING_AGGREGATION_PATH}")

90,693 OOF slices over 361 patients (slices/patient: min=128, median=232, max=526)
Fold-disjointness and label-consistency checks passed.



     granularity                                     rule  n_units  roc_auc  precision  recall     f1  brier confusion_matrix_tn_fp_fn_tp
slice (baseline) none -- every slice scored independently    90693   0.9920     0.9887  0.9686 0.9786 0.0282         17821/795/2262/69815
volume (patient)                                     mean      361   0.9973     0.9928  0.9858 0.9893 0.0142                   78/2/4/277
volume (patient)                                   median      361   0.9975     0.9928  0.9822 0.9875 0.0146                   78/2/5/276
volume (patient)                                      max      361   0.9913     0.9272  0.9964 0.9605 0.0577                  58/22/1/280
volume (patient)                        top-5%-slice-mean      361   0.9974     0.9428  0.9964 0.9689 0.0449                  63/17/1/280
volume (patient)                        top-10-slice-mean      361   0.9976     0.9396  0.9964 0.9672 0.0458                  62/18/1/280
volume (patient)                  

In [6]:
# Diagnostic: how much do a single patient's per-slice scores actually vary?
cancer_spread = imaging_oof[imaging_oof["y_true"] == 1].groupby("patient_id")["p"].agg(["mean", "min", "std"])
healthy_spread = imaging_oof[imaging_oof["y_true"] == 0].groupby("patient_id")["p"].agg(["mean", "max"])

print("=== WITHIN-PATIENT SPREAD (diagnostic, not a performance metric) ===\n")
print(f"Cancer patients (n={len(cancer_spread)}) -- only a MINORITY of each patient's slices contain tumour,")
print("yet every slice carries the patient's cancer label:")
print(f"  mean per-patient MEAN slice prob : {cancer_spread['mean'].mean():.3f}")
print(f"  mean per-patient MIN  slice prob : {cancer_spread['min'].mean():.3f}  <- would be LOW if the model localized tumour")
print(f"  mean per-patient STD             : {cancer_spread['std'].mean():.3f}  <- low STD = 'cancer' on ~every slice")
print(f"  patients whose LOWEST slice still scores >0.5: {(cancer_spread['min'] > 0.5).mean():.1%}")

print(f"\nHealthy patients (n={len(healthy_spread)}):")
print(f"  mean per-patient MEAN slice prob : {healthy_spread['mean'].mean():.3f}")
print(f"  mean per-patient MAX  slice prob : {healthy_spread['max'].mean():.3f}")
print(f"  patients with ANY slice scoring >0.5: {(healthy_spread['max'] > 0.5).mean():.1%}  <- why max/top-k aggregation loses precision")

print("\nINTERPRETATION: near-flat within-patient scores mean the cancer signal is GLOBAL to the volume,")
print("not localized to tumour slices -- consistent with the disclosed confound below. This is why 'mean'")
print("outperforms max/top-k here, and why volume aggregation improves the numbers WITHOUT resolving the")
print("underlying tumour-under-reliance finding.")
print("\n" + textwrap.fill(IMAGING_CAVEAT, 100))

=== WITHIN-PATIENT SPREAD (diagnostic, not a performance metric) ===

Cancer patients (n=281) -- only a MINORITY of each patient's slices contain tumour,
yet every slice carries the patient's cancer label:
  mean per-patient MEAN slice prob : 0.973
  mean per-patient MIN  slice prob : 0.424  <- would be LOW if the model localized tumour
  mean per-patient STD             : 0.053  <- low STD = 'cancer' on ~every slice
  patients whose LOWEST slice still scores >0.5: 35.6%

Healthy patients (n=80):
  mean per-patient MEAN slice prob : 0.132
  mean per-patient MAX  slice prob : 0.348
  patients with ANY slice scoring >0.5: 27.5%  <- why max/top-k aggregation loses precision

INTERPRETATION: near-flat within-patient scores mean the cancer signal is GLOBAL to the volume,
not localized to tumour slices -- consistent with the disclosed confound below. This is why 'mean'
outperforms max/top-k here, and why volume aggregation improves the numbers WITHOUT resolving the
underlying tumour-under-re

## Per-Branch Inference Functions + the Combiner

In [7]:
SLICE_AGGREGATORS = {"mean": np.mean, "median": np.median, "max": np.max}


def run_tabular_branch(patient_features: dict) -> float:
    """patient_features: dict keyed by TABULAR_FEATURES (plasma_CA19_9 may be NaN/missing).
    Returns the clinical branch's own calibrated PDAC probability -- nothing imaging-related here."""
    df = pd.DataFrame([patient_features])[TABULAR_FEATURES]
    df_imputed = clinical_imputer.transform(df)
    raw_proba = clinical_model.predict_proba(df_imputed[TABULAR_FEATURES])[:, 1]
    calibrated = clinical_calibrator.predict_proba(raw_proba.reshape(-1, 1))[:, 1]
    return float(calibrated[0])


def _score_slices(slice_stack: torch.Tensor) -> np.ndarray:
    """slice_stack: (N, 3, BOX, BOX). Returns (N,) calibrated per-slice probabilities.
    Batched so a full ~250-slice volume never lands on the GPU in one go."""
    raw_batches = []
    with torch.no_grad():
        for start in range(0, len(slice_stack), IMAGING_VOLUME_BATCH_SIZE):
            batch = slice_stack[start:start + IMAGING_VOLUME_BATCH_SIZE].to(DEVICE)
            _, det_logit = imaging_model(batch)
            raw_batches.append(torch.sigmoid(det_logit).cpu().numpy().reshape(-1))
    raw_proba = np.concatenate(raw_batches)
    return imaging_calibrator.predict_proba(raw_proba.reshape(-1, 1))[:, 1]


def run_imaging_branch(image_tensor: torch.Tensor) -> float:
    """image_tensor: (3, BOX, BOX) float32 in [0, 1], already 3-channel-replicated -- one
    preprocessed CT slice, matching SliceCacheDataset's output. Returns that slice's calibrated
    probability. This is the imaging model's NATIVE granularity (how it was trained/evaluated)."""
    return float(_score_slices(image_tensor.unsqueeze(0))[0])


def run_imaging_branch_volume(slice_stack: torch.Tensor, aggregation: str = None) -> dict:
    """slice_stack: (N, 3, BOX, BOX) -- a whole CT scan's preprocessed slices, in order.
    Returns the aggregated patient-level score AND the full per-slice vector (the dashboard
    needs the profile, and it's also how the model's flat within-patient response stays visible).

    Aggregation defaults to the fixed IMAGING_SLICE_AGGREGATION rule; the parameter exists so the
    measurement cell's alternatives stay reachable, not so callers can tune it per patient."""
    aggregation = aggregation or IMAGING_SLICE_AGGREGATION
    if aggregation not in SLICE_AGGREGATORS:
        raise ValueError(f"Unknown aggregation '{aggregation}'; expected one of {sorted(SLICE_AGGREGATORS)}")
    if slice_stack.ndim != 4:
        raise ValueError(f"Expected a (N, 3, BOX, BOX) volume, got shape {tuple(slice_stack.shape)}")

    per_slice = _score_slices(slice_stack)
    return {
        "patient_score": float(SLICE_AGGREGATORS[aggregation](per_slice)),
        "per_slice_proba": per_slice,
        "n_slices": int(len(per_slice)),
        "aggregation": aggregation,
    }


def fuse(imaging_input: torch.Tensor = None, tabular_input: dict = None) -> dict:
    """imaging_input may be a single slice (3, BOX, BOX) or a whole volume (N, 3, BOX, BOX) --
    dispatched on tensor rank, and which one was used is recorded in the result. At least one of
    imaging_input / tabular_input must be given; never fabricates a joint score when only one
    branch's input is available (rule 3)."""
    if imaging_input is None and tabular_input is None:
        raise ValueError("fuse() requires at least one of imaging_input or tabular_input.")

    result = {"imaging_calibrated_proba": None, "imaging_granularity": None, "imaging_n_slices": None,
              "tabular_calibrated_proba": None, "fused_score": None, "mode": None}

    if imaging_input is not None:
        if imaging_input.ndim == 3:
            result["imaging_calibrated_proba"] = run_imaging_branch(imaging_input)
            result["imaging_granularity"] = "single-slice"
            result["imaging_n_slices"] = 1
        elif imaging_input.ndim == 4:
            volume_result = run_imaging_branch_volume(imaging_input)
            result["imaging_calibrated_proba"] = volume_result["patient_score"]
            result["imaging_granularity"] = f"volume ({volume_result['aggregation']} over slices)"
            result["imaging_n_slices"] = volume_result["n_slices"]
        else:
            raise ValueError(
                f"imaging_input must be (3, BOX, BOX) or (N, 3, BOX, BOX), got {tuple(imaging_input.shape)}"
            )

    if tabular_input is not None:
        result["tabular_calibrated_proba"] = run_tabular_branch(tabular_input)

    if imaging_input is not None and tabular_input is not None:
        result["fused_score"] = (
            W_IMAGING * result["imaging_calibrated_proba"] + W_TABULAR * result["tabular_calibrated_proba"]
        )
        result["mode"] = "fused (both modalities)"
    elif imaging_input is not None:
        result["fused_score"] = result["imaging_calibrated_proba"]
        result["mode"] = "single-modality (imaging only)"
    else:
        result["fused_score"] = result["tabular_calibrated_proba"]
        result["mode"] = "single-modality (tabular only)"

    return result


print("run_tabular_branch, run_imaging_branch, run_imaging_branch_volume, fuse() defined.")

run_tabular_branch, run_imaging_branch, run_imaging_branch_volume, fuse() defined.


## Real Sample Inputs for the Smoke Test

In [8]:
def load_slices(cache_rows: pd.DataFrame) -> torch.Tensor:
    """cache_rows: manifest_cache rows (already ordered). Returns (N, 3, BOX, BOX) float32 in [0,1],
    preprocessed exactly as SliceCacheDataset.__getitem__ does."""
    imgs = np.stack([
        np.asarray(images_memmap[int(r)], dtype=np.float32) / 255.0 for r in cache_rows["img_row"]
    ])
    return torch.from_numpy(np.repeat(imgs[:, None, :, :], 3, axis=1).copy())


# Real tabular sample -- a real patient with a missing plasma_CA19_9 value, exercising the imputer path
tabular_clean_df = pd.read_csv(TABULAR_CLEAN_PATH)
sample_tabular_row = tabular_clean_df[tabular_clean_df["plasma_CA19_9"].isna()].iloc[0]
sample_tabular_input = sample_tabular_row[TABULAR_FEATURES].to_dict()

print(f"Real tabular sample: patient {sample_tabular_row['sample_id']} "
      f"(true label: {'PDAC' if sample_tabular_row['target_binary'] else 'not PDAC'}, plasma_CA19_9 missing)")
print(sample_tabular_input)

manifest_cache_df = pd.read_csv(CACHE_MANIFEST_PATH)
images_memmap = np.load(CACHE_IMAGES_PATH, mmap_mode="r")

# Real imaging sample 1 -- a single real cancer-class MSD slice (native granularity)
sample_slice_row = manifest_cache_df[manifest_cache_df["class"] == 1].iloc[0]
sample_imaging_input = load_slices(manifest_cache_df.loc[[sample_slice_row.name]])[0]

print(f"\nReal single-slice sample: {sample_slice_row['patient_id']} slice {sample_slice_row['slice_index']} "
      f"({sample_slice_row['dataset']}, true class={sample_slice_row['class']}), "
      f"tensor shape {tuple(sample_imaging_input.shape)}")

# Real imaging sample 2 -- that same patient's ENTIRE scan, in slice order (volume granularity)
sample_volume_patient = sample_slice_row["patient_id"]
sample_volume_rows = manifest_cache_df[
    manifest_cache_df["patient_id"] == sample_volume_patient
].sort_values("slice_index")
sample_imaging_volume = load_slices(sample_volume_rows)

print(f"Real volume sample:       {sample_volume_patient} -- all {len(sample_volume_rows)} slices "
      f"({sample_volume_rows['dataset'].iloc[0]}, true class={sample_volume_rows['class'].iloc[0]}), "
      f"tensor shape {tuple(sample_imaging_volume.shape)}")

Real tabular sample: patient S10 (true label: not PDAC, plasma_CA19_9 missing)
{'creatinine': 0.97266, 'LYVE1': 2.037585, 'REG1B': 94.46703, 'TFF1': 209.48825, 'plasma_CA19_9': nan, 'age': 81, 'sex': 0}



Real single-slice sample: pancreas_001 slice 0 (MSD, true class=1), tensor shape (3, 320, 320)


Real volume sample:       pancreas_001 -- all 275 slices (MSD, true class=1), tensor shape (275, 3, 320, 320)


## Smoke Test: Every Real Input Path

In [9]:
result_imaging_slice_only = fuse(imaging_input=sample_imaging_input, tabular_input=None)
result_imaging_volume_only = fuse(imaging_input=sample_imaging_volume, tabular_input=None)
result_tabular_only = fuse(imaging_input=None, tabular_input=sample_tabular_input)
result_both = fuse(imaging_input=sample_imaging_volume, tabular_input=sample_tabular_input)

for name, result in [
    ("IMAGING-ONLY (single slice)", result_imaging_slice_only),
    ("IMAGING-ONLY (whole volume)", result_imaging_volume_only),
    ("TABULAR-ONLY", result_tabular_only),
    ("BOTH: volume + tabular (fabricated pairing -- code-path smoke test only, not a real patient)", result_both),
]:
    print(f"=== {name} ===")
    print(json.dumps(result, indent=2))
    print()

# Rule 3: single-modality paths pass the branch's own score through untouched
for single in (result_imaging_slice_only, result_imaging_volume_only):
    assert single["tabular_calibrated_proba"] is None
    assert single["fused_score"] == single["imaging_calibrated_proba"]
    assert single["mode"] == "single-modality (imaging only)"

assert result_tabular_only["imaging_calibrated_proba"] is None
assert result_tabular_only["imaging_granularity"] is None
assert result_tabular_only["fused_score"] == result_tabular_only["tabular_calibrated_proba"]
assert result_tabular_only["mode"] == "single-modality (tabular only)"

# Granularity is recorded explicitly, never left implicit
assert result_imaging_slice_only["imaging_granularity"] == "single-slice"
assert result_imaging_slice_only["imaging_n_slices"] == 1
assert result_imaging_volume_only["imaging_granularity"] == f"volume ({IMAGING_SLICE_AGGREGATION} over slices)"
assert result_imaging_volume_only["imaging_n_slices"] == len(sample_volume_rows)

# Fused path applies exactly the stated weights
assert result_both["mode"] == "fused (both modalities)"
expected_fused = W_IMAGING * result_both["imaging_calibrated_proba"] + W_TABULAR * result_both["tabular_calibrated_proba"]
assert abs(result_both["fused_score"] - expected_fused) < 1e-9

for result in (result_imaging_slice_only, result_imaging_volume_only, result_tabular_only, result_both):
    for key in ("imaging_calibrated_proba", "tabular_calibrated_proba", "fused_score"):
        value = result[key]
        assert value is None or 0.0 <= value <= 1.0, f"{key}={value} out of [0, 1]"

# The per-slice profile the dashboard will render
volume_detail = run_imaging_branch_volume(sample_imaging_volume)
per_slice = volume_detail["per_slice_proba"]
assert len(per_slice) == len(sample_volume_rows)
assert abs(volume_detail["patient_score"] - float(np.mean(per_slice))) < 1e-9
print(f"Per-slice profile for {sample_volume_patient}: n={len(per_slice)}, "
      f"min={per_slice.min():.4f}, mean={per_slice.mean():.4f}, max={per_slice.max():.4f}, std={per_slice.std():.4f}")

print("\nAll smoke-test assertions passed: 4 real input paths ran with no exceptions, every probability "
      "in [0, 1], single-modality paths pass their branch's score through untouched, granularity recorded "
      "explicitly, fused score matches the stated weights.")

print("\nIMAGING CAVEAT applies to every imaging_calibrated_proba / fused_score above where imaging contributed:")
print(textwrap.fill(IMAGING_CAVEAT, 100))

=== IMAGING-ONLY (single slice) ===
{
  "imaging_calibrated_proba": 0.9946813853742091,
  "imaging_granularity": "single-slice",
  "imaging_n_slices": 1,
  "tabular_calibrated_proba": null,
  "fused_score": 0.9946813853742091,
  "mode": "single-modality (imaging only)"
}

=== IMAGING-ONLY (whole volume) ===
{
  "imaging_calibrated_proba": 0.9913239006929702,
  "imaging_granularity": "volume (mean over slices)",
  "imaging_n_slices": 275,
  "tabular_calibrated_proba": null,
  "fused_score": 0.9913239006929702,
  "mode": "single-modality (imaging only)"
}

=== TABULAR-ONLY ===
{
  "imaging_calibrated_proba": null,
  "imaging_granularity": null,
  "imaging_n_slices": null,
  "tabular_calibrated_proba": 0.13006893783334378,
  "fused_score": 0.13006893783334378,
  "mode": "single-modality (tabular only)"
}

=== BOTH: volume + tabular (fabricated pairing -- code-path smoke test only, not a real patient) ===
{
  "imaging_calibrated_proba": 0.9913239006929702,
  "imaging_granularity": "volume 

Per-slice profile for pancreas_001: n=275, min=0.1629, mean=0.9913, max=0.9947, std=0.0501

All smoke-test assertions passed: 4 real input paths ran with no exceptions, every probability in [0, 1], single-modality paths pass their branch's score through untouched, granularity recorded explicitly, fused score matches the stated weights.

IMAGING CAVEAT applies to every imaging_calibrated_proba / fused_score above where imaging contributed:
KNOWN LIMITATION (checkpoints/imaging/final/model_card.json, confound-check investigation): A
rigorous, 6-round confound-check investigation (Grad-CAM + 5 other attribution methods, then causal
occlusion testing) found this architecture's detection head originally relied MORE on the BOX=320
packing's synthetic padding than on the real tumour region (Round 4: 2.03x a random-patch control,
p=0.044). A random-resized-crop augmentation (augment=True, used in this final fit) verified-fixed
the padding shortcut (Round 5: 1.08x control, p=0.917 -- statistica

## Fourth Path: Neither Input Given

In [10]:
try:
    fuse(imaging_input=None, tabular_input=None)
    raise AssertionError("Expected fuse() to raise ValueError when given no input")
except ValueError as e:
    print(f"Correctly raised ValueError when no input was given: {e}")

Correctly raised ValueError when no input was given: fuse() requires at least one of imaging_input or tabular_input.


## Write the Fusion-Level `checkpoints/fusion/final/model_card.json`

In [11]:
aggregation_measurement = (
    aggregation_df.set_index(["granularity", "rule"])
    [["n_units", "roc_auc", "precision", "recall", "f1", "brier", "confusion_matrix_tn_fp_fn_tp"]]
    .round(4).reset_index().to_dict(orient="records")
)

fusion_model_card = {
    "stage": "fusion",
    "created_on": date.today().isoformat(),
    "combination_rule": {
        "type": "weighted_average_of_calibrated_probabilities",
        "formula": "fused = W_IMAGING * imaging_calibrated_proba + W_TABULAR * tabular_calibrated_proba",
        "weights": {"W_IMAGING": W_IMAGING, "W_TABULAR": W_TABULAR},
        "reasoning": COMBINATION_RULE_REASONING,
    },
    "imaging_slice_aggregation_rule": {
        "rule": IMAGING_SLICE_AGGREGATION,
        "applies_to": (
            "Whole-CT-volume input only. A single-slice input path (the imaging model's native "
            "granularity, how it was trained and evaluated) remains available and is unaffected."
        ),
        "reasoning": IMAGING_SLICE_AGGREGATION_REASONING,
        "measurement_source": "results/fusion/imaging_slice_vs_volume_aggregation.csv",
        "measurement": aggregation_measurement,
        "measurement_caveat": (
            "Measured from the imaging branch's real 5-fold OOF slice predictions, aggregated per "
            "patient (verified fold-disjoint, so no train/held-out mixing). These patient-level "
            "numbers are NOT cleaner evidence than the slice-level ones: dataset perfectly predicts "
            "class in this project (all MSD patients cancer, all NIH healthy), so they inherit the "
            "same scanner-confound at a coarser grain. No model was fit or tuned to produce them."
        ),
    },
    "single_modality_behavior": (
        "If only one modality's input is provided, fuse() returns that branch's own calibrated "
        "score directly as fused_score, labeled mode='single-modality (<branch> only)' -- never "
        "silently degraded or combined with a fabricated placeholder for the missing branch."
    ),
    "no_joint_benchmark_metric": (
        "No joint/paired accuracy, precision, recall, F1, ROC-AUC, or confusion matrix exists for "
        "this combiner, and none is computed anywhere in this project. No patient in this project "
        "has both a CT scan and a urine sample (MSD, NIH, and the Debernardi et al. urine cohort are "
        "three separate, unpaired sources), so there is no ground truth against which a fused "
        "prediction could be scored. The weights above are a stated judgment call, not something fit "
        "or validated against held-out fused performance -- see results/fusion/pair_comparison.csv "
        "and src/fusion/fusion_evaluation.ipynb for each branch's own, separately-computed metrics. "
        "The imaging_slice_aggregation_rule measurement is a SINGLE-BRANCH (imaging-only) "
        "measurement and is not, and must not be presented as, a joint/fused benchmark."
    ),
    "branch_model_cards": {
        "clinical": "checkpoints/clinical/final/model_card.json",
        "imaging": "checkpoints/imaging/final/model_card.json",
    },
    "imaging_known_limitations_caveat": IMAGING_CAVEAT,
    "smoke_test": {
        "imaging_only_single_slice": result_imaging_slice_only,
        "imaging_only_whole_volume": result_imaging_volume_only,
        "tabular_only": result_tabular_only,
        "both_fabricated_pairing_not_a_real_patient": result_both,
    },
}

with open(CHECKPOINTS_FUSION_FINAL_DIR / "model_card.json", "w") as f:
    json.dump(fusion_model_card, f, indent=2)

print(f"Saved {CHECKPOINTS_FUSION_FINAL_DIR / 'model_card.json'}")

Saved C:\FYP\checkpoints\fusion\final\model_card.json


## Summary

In [12]:
with open(CHECKPOINTS_FUSION_FINAL_DIR / "model_card.json") as f:
    saved_card = json.load(f)

assert saved_card["imaging_known_limitations_caveat"] == IMAGING_CAVEAT
assert saved_card["combination_rule"]["weights"] == {"W_IMAGING": W_IMAGING, "W_TABULAR": W_TABULAR}
assert saved_card["imaging_slice_aggregation_rule"]["rule"] == IMAGING_SLICE_AGGREGATION
assert len(saved_card["imaging_slice_aggregation_rule"]["measurement"]) == len(aggregation_df)

print("=== SUMMARY ===\n")
print("Fixed rules:")
print(f"  cross-modality : fused = {W_IMAGING} * imaging_calibrated_proba + {W_TABULAR} * tabular_calibrated_proba")
print(f"  imaging volume : patient_score = {IMAGING_SLICE_AGGREGATION}(per-slice calibrated probabilities)\n")

print("Slice vs. volume measurement (imaging branch only -- NOT a joint benchmark):")
print(aggregation_df[["granularity", "rule", "n_units", "roc_auc", "precision", "recall", "f1", "brier"]]
      .to_string(index=False, float_format=lambda v: f"{v:.4f}"))

print("\nSmoke test (4 real input paths, all passed):")
for label, r in [
    ("imaging (slice) ", result_imaging_slice_only),
    ("imaging (volume)", result_imaging_volume_only),
    ("tabular         ", result_tabular_only),
    ("both            ", result_both),
]:
    granularity = r["imaging_granularity"] or "n/a"
    print(f"  {label} : fused_score={r['fused_score']:.4f}  mode={r['mode']:<32} imaging_granularity={granularity}")
print("  (neither)        : correctly raised ValueError\n")

print(f"Aggregation measurement saved to {FUSION_IMAGING_AGGREGATION_PATH}")
print(f"Fusion-level model_card.json saved and verified at {CHECKPOINTS_FUSION_FINAL_DIR / 'model_card.json'}")
print("Confirmed: imaging_known_limitations_caveat present and matches IMAGING_CAVEAT verbatim; both fixed")
print("rules and the full aggregation measurement round-tripped correctly.\n")

print("IMAGING CAVEAT (carried forward into the fusion-level model card):")
print(textwrap.fill(IMAGING_CAVEAT, 100))

=== SUMMARY ===

Fixed rules:
  cross-modality : fused = 0.4 * imaging_calibrated_proba + 0.6 * tabular_calibrated_proba
  imaging volume : patient_score = mean(per-slice calibrated probabilities)

Slice vs. volume measurement (imaging branch only -- NOT a joint benchmark):
     granularity                                     rule  n_units  roc_auc  precision  recall     f1  brier
slice (baseline) none -- every slice scored independently    90693   0.9920     0.9887  0.9686 0.9786 0.0282
volume (patient)                                     mean      361   0.9973     0.9928  0.9858 0.9893 0.0142
volume (patient)                                   median      361   0.9975     0.9928  0.9822 0.9875 0.0146
volume (patient)                                      max      361   0.9913     0.9272  0.9964 0.9605 0.0577
volume (patient)                        top-5%-slice-mean      361   0.9974     0.9428  0.9964 0.9689 0.0449
volume (patient)                        top-10-slice-mean      361   0.